# Capítulo 3 · Actividad: limpieza de datos y outliers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_03/02_actividad_limpieza_outliers.ipynb)

Vas a aplicar lo que viste en la notebook de clase `04_limpieza_y_outliers` a **tres datasets distintos**. Cada uno tiene problemas diferentes, igual que en la vida real.

### Antes de empezar

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia.
2. Si todavía no tienes los datos, descárgalos desde la raíz del repo:
   ```bash
   uv run python scripts/download_data.py
   ```

### Cómo trabajar

- Cada ejercicio tiene una celda `# Tu código aquí`. Añade todas las celdas que necesites.
- En las celdas ✍️ **justifica tus decisiones**: en limpieza de datos casi nunca hay una única respuesta correcta, pero siempre tiene que haber un motivo.
- Al terminar, ejecuta **Kernel → Restart & Run All**.

| Parte | Datos | Tema |
|---|---|---|
| A. Empleados | una tabla pequeña creada en la notebook | Limpieza estructural e imputación |
| B. Pingüinos | `data/penguins.csv` | Valores faltantes |
| C. Viviendas de California | `data/california_housing.csv` | Outliers |

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.impute import SimpleImputer

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"

---
## Parte A · Empleados 👩‍💼

Recursos Humanos te pasa esta tabla, exportada a mano de varias hojas de cálculo. Ejecuta la celda para crearla.

In [ ]:
empleados = pd.DataFrame({
    "nombre": ["Ana", "Luis", "Marta", "Luis", "Pedro", "Sofía", "Juan", "Elena",
               "Diego", "Carla", "Raúl", "Elena", "Iván", "Lucía"],
    "departamento": [" ventas", "IT", "it ", "IT", "Ventas", "RRHH", "rrhh", "VENTAS",
                     "It", "ventas ", "IT", "VENTAS", "RRHH", " Rrhh"],
    "edad": [34, np.nan, 29, np.nan, 45, 38, np.nan, 41, 27, np.nan, 52, 41, 31, 36],
    "salario": ["$35,000", "$52,000", "$48,500", "$52,000", "39000", "$41,200", "$37,800",
                "40500", "$47,000", "$36,900", "$61,000", "40500", "$39,900", "$42,300"],
    "fecha_ingreso": ["2019-03-15", "2020-07-01", "2021-01-10", "2020-07-01", "2015-09-30",
                      "2018-11-05", "2022-02-14", "2016-06-20", "2023-04-03", "2021-08-16",
                      "2012-01-09", "2016-06-20", "2019-10-21", "2017-05-02"],
})
empleados

### Ejercicio 1 · Diagnóstico

Haz el diagnóstico de calidad de `empleados`: tipos de cada columna, valores faltantes, filas duplicadas y cómo están escritas las categorías.

In [ ]:
empleados.info()
print("duplicados:", empleados.duplicated().sum())
print("\nfaltantes:\n", empleados.isna().sum())
print("\ndepartamento:\n", empleados["departamento"].value_counts())

✍️ **Pregunta:** haz una lista de todos los problemas que encontraste.

✍️ **Tu respuesta:**
al revisar la tabla encontre varios problemas:
- faltan datos: en la columna edad hay 4 valores nulos.
- filas duplicadas: hay registros repetidos (luis de it y elena de ventas).
- texto desordenado: la columna departamento tiene mayusculas, minusculas y espacios en blanco de mas (como ' ventas', 'IT', 'it ').
- tipos de datos incorrectos: el salario esta como texto con el signo $ y comas, y la fecha de ingreso tambien es texto en vez de datetime.

### Ejercicio 2 · Duplicados

Elimina las filas duplicadas. ¿Cuántas filas quedan?

In [ ]:
empleados = empleados.drop_duplicates().reset_index(drop=True)
print("filas restantes:", len(empleados))
empleados

### Ejercicio 3 · Texto inconsistente

Normaliza `departamento` para que queden **exactamente tres** valores: `Ventas`, `IT` y `RRHH` (las siglas, en mayúsculas).

In [ ]:
# quitamos espacios y unificamos nombres
empleados["departamento"] = empleados["departamento"].str.strip()

mapeo = {"ventas": "Ventas", "it": "IT", "rrhh": "RRHH"}
empleados["departamento"] = empleados["departamento"].str.lower().map(mapeo)

empleados["departamento"].value_counts()

### Ejercicio 4 · Tipos de datos

a) Convierte `salario` a número. Ojo: no todos los valores tienen el mismo formato.

b) Convierte `fecha_ingreso` a fecha y crea la columna `año_ingreso`.

c) Comprueba los tipos con `dtypes`.

In [ ]:
# a) salario a numero
empleados["salario"] = empleados["salario"].str.replace("$", "", regex=False).str.replace(",", "", regex=False)
empleados["salario"] = pd.to_numeric(empleados["salario"])

# b) fecha de ingreso y año
empleados["fecha_ingreso"] = pd.to_datetime(empleados["fecha_ingreso"])
empleados["año_ingreso"] = empleados["fecha_ingreso"].dt.year

# c) comprobar tipos
empleados.dtypes

### Ejercicio 5 · Imputación

a) Crea una columna `edad_faltante` que indique qué empleados no tenían la edad.

b) Rellena la edad con la **mediana de su departamento**.

c) Comprueba que ya no quedan nulos y muestra la tabla final.

In [ ]:
# a) columna indicadora
empleados["edad_faltante"] = empleados["edad"].isna().astype(int)

# b) imputar con mediana por departamento
mediana_depto = empleados.groupby("departamento")["edad"].transform("median")
empleados["edad"] = empleados["edad"].fillna(mediana_depto)

# c) comprobar nulos y ver tabla
print("nulos restantes:", empleados.isna().sum().sum())
empleados

✍️ **Pregunta:** ¿por qué la mediana del departamento y no la mediana de todos los empleados? ¿En qué caso daría igual?

✍️ **Tu respuesta:**
porque en cada departamento la gente suele tener edades diferentes (por ejemplo perfiles mas jovenes o con mas años de experiencia segun el area), asi que imputar por grupo se acerca mas a la realidad.

daria lo mismo si la mediana de edad fuera igual en todas las areas o si la edad no tuviera ninguna relacion con el departamento.

---
## Parte B · Pingüinos 🐧 (valores faltantes)

El archivo `data/penguins.csv` tiene medidas de 344 pingüinos de tres especies. Columnas: `species`, `island`, `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g` y `sex`.

### Ejercicio 6 · Diagnóstico de faltantes

a) Carga los datos en `pinguinos`.

b) Calcula cuántos valores faltan en cada columna y qué porcentaje representan.

c) Muestra las filas que tienen algún valor faltante.

In [ ]:
# a) cargar datos
pinguinos = pd.read_csv(DATA_DIR / "penguins.csv")

# b) faltantes y porcentaje
faltantes = pd.DataFrame({
    "nulos": pinguinos.isna().sum(),
    "porcentaje": (pinguinos.isna().mean() * 100).round(1)
})
display(faltantes)

# c) filas con algun faltante
pinguinos[pinguinos.isna().any(axis=1)]

✍️ **Pregunta:** ¿ves algún patrón en las filas con faltantes? ¿Faltan los datos de forma parecida en todas?

✍️ **Tu respuesta:**
si, se nota un patron claro:
- hay dos pinguinos (filas 3 y 339) a los que les faltan todas las medidas del cuerpo y tampoco tienen sexo.
- en el resto de las filas con nulos, las medidas del cuerpo si estan completas y lo unico que falta es la variable sexo.
no faltan igual en todas: dos filas estan casi vacias y las demas solo no tienen registrado el sexo.

### Ejercicio 7 · Eliminar o imputar

Hay pingüinos a los que les faltan **todas** las medidas. Elimínalos y comprueba cuántas filas quedan.

In [ ]:
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
pinguinos = pinguinos.dropna(subset=medidas, how="all").reset_index(drop=True)

print("filas restantes:", len(pinguinos))
pinguinos.isna().sum()

✍️ **Pregunta:** ¿por qué en este caso es mejor eliminar esas filas que imputarlas?

✍️ **Tu respuesta:**
porque a esas dos filas les falta toda la informacion de las medidas (pico, aletas, peso). si intentaramos rellenarlas estariamos practicamente inventando un pinguino de la nada solo con su especie e isla. ademas solo son 2 filas entre 344, asi que borrarlas no afecta casi nada a la muestra.

### Ejercicio 8 · Dos estrategias para el sexo

Compara dos formas de rellenar `sex`, cada una en una **copia** de `pinguinos`:

a) Con la **moda**.

b) Con una categoría nueva, `"desconocido"`.

Muestra `value_counts()` del resultado en cada caso.

In [ ]:
# a) imputar con la moda
pinguinos_moda = pinguinos.copy()
moda_sex = pinguinos_moda["sex"].mode()[0]
pinguinos_moda["sex"] = pinguinos_moda["sex"].fillna(moda_sex)
print("con la moda:\n", pinguinos_moda["sex"].value_counts())

# b) imputar con 'desconocido'
pinguinos_desconocido = pinguinos.copy()
pinguinos_desconocido["sex"] = pinguinos_desconocido["sex"].fillna("desconocido")
print("\ncon 'desconocido':\n", pinguinos_desconocido["sex"].value_counts())

✍️ **Pregunta:** ¿qué estrategia elegirías aquí y por qué? Fíjate en cuántos machos y hembras hay.

✍️ **Tu respuesta:**
yo elegiria ponerle 'desconocido'. la razon es que machos y hembras estan muy balanceados (168 machos y 165 hembras). si rellenamos todos los faltantes con la moda que es macho, estariamos inclinando la balanza sin estar seguros y creando un sesgo artificial.

### Ejercicio 9 · Con scikit-learn

Usa `SimpleImputer` para rellenar `sex` con el valor más frecuente. ¿Qué valor aprendió? ¿Coincide con la moda del ejercicio 8?

In [ ]:
imputer = SimpleImputer(strategy="most_frequent")
imputer.fit(pinguinos[["sex"]])

valor_aprendido = imputer.statistics_[0]
print("valor aprendido:", valor_aprendido)
print("coincide con la moda:", valor_aprendido == moda_sex)

---
## Parte C · Viviendas de California 🏠 (outliers)

El archivo `data/california_housing.csv` viene del censo de California de 1990. **Cada fila es un distrito** (un barrio), no una vivienda:

| Columna | Significado |
|---|---|
| `MedInc` | ingreso mediano del distrito (en decenas de miles de $) |
| `HouseAge` | antigüedad mediana de las viviendas (años) |
| `AveRooms` | número medio de habitaciones por vivienda |
| `AveBedrms` | número medio de dormitorios por vivienda |
| `Population` | población del distrito |
| `AveOccup` | número medio de personas por vivienda |
| `Latitude`, `Longitude` | ubicación |
| `MedHouseVal` | valor mediano de las viviendas (en cientos de miles de $) |

### Ejercicio 10 · Primer vistazo

Carga los datos en `california` y muestra sus estadísticas descriptivas.

In [ ]:
california = pd.read_csv(DATA_DIR / "california_housing.csv")
california.describe()

✍️ **Pregunta:** ¿en qué columnas el máximo está muy lejos del percentil 75%? ¿Te parece posible algún valor?

✍️ **Tu respuesta:**
las columnas donde el maximo esta lejismos del percentil 75% son:
- aveoccup: el 75% es como 3.28 personas por casa y el maximo llega a 1243 personas.
- averooms: el 75% es alrededor de 6 cuartos y el maximo es de 142.
- avebedrms: el 75% es 1.1 y el maximo sube a 34 cuartos.
- population: el 75% es 1725 y el maximo pasa los 35,000 habitantes.

un promedio de 1243 personas o 142 cuartos por vivienda en un barrio no es realista, son errores claros al capturar los datos. la poblacion de 35,000 quiza sea posible si el distrito es muy grande, aunque sigue siendo un valor muy extremo.

### Ejercicio 11 · Detección visual

Dibuja en una misma figura el boxplot de `AveRooms`, `AveOccup` y `Population`, cada uno en su propio gráfico.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

sns.boxplot(y=california["AveRooms"], ax=axes[0])
axes[0].set_title("AveRooms")

sns.boxplot(y=california["AveOccup"], ax=axes[1])
axes[1].set_title("AveOccup")

sns.boxplot(y=california["Population"], ax=axes[2])
axes[2].set_title("Population")

plt.tight_layout()
plt.show()

### Ejercicio 12 · Z-score contra IQR

Para **cada columna** de `california`, cuenta cuántos outliers detecta el z-score ($|z| > 3$) y cuántos el IQR. Junta los resultados en un DataFrame con una fila por columna.

In [ ]:
# contamos outliers por z-score e iqr para cada columna
resumen = []
for col in california.columns:
    # z-score
    z = (california[col] - california[col].mean()) / california[col].std()
    n_zscore = (z.abs() > 3).sum()

    # iqr
    q1, q3 = california[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_iqr = ((california[col] < q1 - 1.5 * iqr) | (california[col] > q3 + 1.5 * iqr)).sum()

    resumen.append({"columna": col, "outliers_zscore": n_zscore, "outliers_iqr": n_iqr})

pd.DataFrame(resumen).set_index("columna")

✍️ **Pregunta:** en `AveOccup`, ¿por qué el z-score detecta tan pocos outliers y el IQR tantos? ¿Cuál de los dos números te parece más razonable?

✍️ **Tu respuesta:**
en aveoccup el z-score detecta muy poquitos porque usa la media y la desviacion estandar. como hay valores exageradamente grandes (como 1243 personas), inflan la desviacion estandar y hacen que la mayoria de los datos raros queden escondidos dentro de las 3 desviaciones (enmascaramiento).

el iqr es mas confiable porque trabaja con cuartiles y los cuartiles no se dejan mover por valores extremos. por eso el resultado del iqr tiene mucho mas sentido.

### Ejercicio 13 · Criterio de negocio

`AveOccup` es el número medio de personas que viven en cada vivienda del distrito.

a) Muestra los 5 distritos con mayor `AveOccup`.

b) Decide un **umbral razonable** por encima del cual el valor no tiene sentido, y elimina esos distritos. ¿Cuántos eliminaste?

In [ ]:
# a) 5 distritos con mayor ocupacion
display(california.nlargest(5, "AveOccup"))

# b) quitamos distritos con promedio mayor a 10 personas
anomalos = california["AveOccup"] > 10
print("distritos eliminados:", anomalos.sum())

california = california[~anomalos].reset_index(drop=True)
print("filas restantes:", len(california))

✍️ **Pregunta:** ¿qué umbral elegiste y por qué?

✍️ **Tu respuesta:**
elegi un limite de 10 personas promedio por casa. en una zona comun de viviendas es muy raro que en promedio cada casa tenga mas de 10 personas, por lo que numeros mayores a eso son casi seguro errores de registro o zonas no residenciales. con este filtro se eliminan 37 casos anomalos sin borrar datos normales.

### Ejercicio 14 · Winsorización

Winsoriza `AveRooms` en los percentiles 1 y 99, guardando el resultado en una columna nueva `AveRooms_wins`. Compara las estadísticas descriptivas de las dos columnas.

In [ ]:
p01, p99 = california["AveRooms"].quantile([0.01, 0.99])
print(f"p01: {p01:.2f} | p99: {p99:.2f}")

california["AveRooms_wins"] = california["AveRooms"].clip(lower=p01, upper=p99)

california[["AveRooms", "AveRooms_wins"]].describe()

### Ejercicio 15 · Un caso raro

Muestra los 5 valores más frecuentes de `MedHouseVal`.

In [ ]:
california["MedHouseVal"].value_counts().head(5)

✍️ **Pregunta:** ¿qué crees que pasó con el valor más repetido? ¿Es un outlier? ¿Qué problema podría causarle a un modelo que intente predecir el valor de las viviendas?

✍️ **Tu respuesta:**
el valor que mas se repite es 5.00001 (que son 500,000 dolares) y sale casi mil veces. esto paso porque los datos tienen un tope maximo: todas las casas que costaban 500,000 dolares o mas se guardaron con ese mismo valor fijo.

no es un outlier comun ni un error de captura, sino que la variable esta truncada por arriba.

el problema para un modelo es que se aplana el precio en el limite superior y el modelo no puede aprender la diferencia entre una casa de 600,000 y una de 2 millones, por lo que va a subestimar el precio de las casas caras.